# Fig. 1 — SEEG event sequences


In [ ]:
# Setup
# Load the raw-data configuration and create an output directory.
import _setup
import numpy as np
from data_processing import raw_to_events as raw
from analysis import from_events as analysis
from plotting import computational as plot
from workflows.tutorial import raw_inputs, TutorialOutput, save_events

config, raw_root = raw_inputs()
out = TutorialOutput('fig1')


In [ ]:
# Fig. 1A: HFO group events
# Read the first raw recording segment; reference and filter the signal.
# Detect HFOs on individual contacts, then combine overlapping detections into group events.
record = config['records'][0]
samples, fs, names = raw.read_samples(raw_root/record['file'],
    start_sec=record.get('start_sec', 0), duration_sec=record['duration_sec'], channels=record.get('channels'))
pre = raw.preprocess(samples, fs, names, config)
detected = raw.detect_groups(pre, config)
out.show(plot.waveform(pre, detected), 'fig1_A_waveforms')


In [ ]:
# Fig. 1B: Event timing
# Compute each participating contact's time centroid using S^3 spectrogram weights.
# Vertical lines mark these centroids within the group-event window.
features = raw.event_features(pre, detected)
out.show(plot.spectra(pre, features, index=0), 'fig1_B_centroids')


In [ ]:
# Fig. 1C: Within-event ranks
# Rank participating contacts from early to late by their time centroids.
# Align contacts by name when combining recording blocks.
# Each column is one event; blank entries indicate nonparticipating contacts.
features['block_ids'] = np.full(features['ranks'].shape[1], record['block_id'])
blocks = [features] + [raw.process_record(r, config, raw_root)[2] for r in config['records'][1:]]
events = raw.join_blocks(blocks)
out.show(plot.rank_heatmap(events), 'fig1_C_ranks')


In [ ]:
# Fig. 1E: Sequence templates
# Cluster normalized participating-contact ranks with K-means and group events by label.
# K=2 defines two sequence templates; it does not establish two physiological states.
clustered = analysis.cluster_events(events, k=2, seed=0)
out.show(plot.rank_heatmap(events, clustered), 'fig1_E_kmeans')


In [ ]:
# Fig. 1D: Sequence consistency
# The matching index measures agreement in contact-pair order.
# Shuffle the order while preserving participation to obtain the null baseline.
stats = analysis.patient_statistics(events, clustered, permutations=200, seed=0)
out.show(plot.statistics(stats, panel='D'), 'fig1_D_statistics')


In [ ]:
# Fig. 1F: Within-template consistency
# Compare matching indices for all events and for each template separately.
# This example contains one patient; cohort statistics require one summary per patient.
out.show(plot.statistics(stats, panel='F'), 'fig1_F_template_statistics')


In [ ]:
# Save events
# Save the computed events and aligned coordinates for Notebook 02.
save_events(out, events, stats, config, raw_root)
